In [26]:
pip install bs4

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ---------- ----------------------------- 1/4 [soupsieve]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   ---------------------------------------- 4/4 [bs4]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests


def test_gs_retrieval_v2():
    base_url = "https://hdpc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"

    # In Oracle HCM, parameters following the 'finder' must be semicolon/comma delimited
    # within the finder string itself for certain 'context' types.
    params = {
        "onlyData": "true",
        "expand": "requisitionList.workLocation",
        "finder": "findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC",
        "limit": 25,
        "offset": 10,
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
        "Accept": "application/json",
        "Referer": "https://houston.gs.com/",  # Some Oracle instances check the referer
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)

        # If it still fails, the API might want the limit inside the finder too
        if response.status_code == 400:
            print("Standard limit failed, trying fallback finder syntax...")
            params["finder"] = (
                "findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC,limit=5"
            )
            del params["limit"]
            response = requests.get(base_url, params=params, headers=headers)

        response.raise_for_status()
        data = response.json()

        jobs = data["items"][0]["requisitionList"]
        print(f"Successfully retrieved {len(jobs)} jobs.")
        for job in jobs:
            print(f"- {job['Title']} ({job['Id']})")

    except Exception as e:
        print(f"Retrieval failed: {e}")
        if "response" in locals():
            print(f"Response Text: {response.text}")


if __name__ == "__main__":
    test_gs_retrieval_v2()

Successfully retrieved 25 jobs.
- Core Data Platform - Software Engineering-Associate-Bengaluru (145662)
- Asset & Wealth Management, Content Marketing Manager, Associate, Dublin (160624)
- Asset & Wealth Management, Social Media, Analyst, Dublin (161799)
- Global Banking & Markets – FICC (Fixed Income, Currencies & Commodities) Engineer - Vice President - London (162477)
- Asset & Wealth Management, Operations MI Analyst, Associate, Dublin (165692)
- Asset & Wealth Management, Business Controls, Associate, Dublin (165691)
- Asset & Wealth Management, Product Manager, Analyst, Dublin (167783)
- Goldman Sachs Asset & Wealth Management - Third Party Wealth Sales - Vice President - Frankfurt (167863)
- Asset & Wealth Management- Client Solutions Group -Data Steward, Analyst/Associate -Bengaluru (168178)
- Asset & Wealth Management- CSG, Sales Enablement, Product Manager- Associate, London (168177)
- Asset & Wealth Management, Alternatives Capital Formation, Commercial Strategy, Associate,

In [ ]:
import requests
import time


def scrape_all_gs_jobs():
    base_url = "https://hdpc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"
    limit = 25  # Keep this at 25 or 50 for stability
    offset = 0
    all_jobs = []

    while True:
        params = {
            "onlyData": "true",
            "expand": "requisitionList.workLocation",
            "finder": f"findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC",
            "limit": limit,
            "offset": offset,
        }

        print(f"Fetching jobs {offset} to {offset + limit}...")
        response = requests.get(base_url, params=params)
        data = response.json()

        # Oracle HCM structure: items[0] contains the data
        root = data["items"][0]
        jobs_in_page = root.get("requisitionList", [])

        if not jobs_in_page:
            break

        all_jobs.extend(jobs_in_page)

        # Check if we've reached the end
        total_available = root.get("TotalJobsCount", 0)
        if len(all_jobs) >= total_available:
            break

        # Move to next page
        offset += limit
        time.sleep(1)  # Be nice to their server!

    print(f"Done! Scraped {len(all_jobs)} total jobs.")
    return all_jobs


# Run it
scrape_all_gs_jobs()

Fetching jobs 0 to 25...
Fetching jobs 25 to 50...
Fetching jobs 50 to 75...
Fetching jobs 75 to 100...
Fetching jobs 100 to 125...
Fetching jobs 125 to 150...
Fetching jobs 150 to 175...
Fetching jobs 175 to 200...
Fetching jobs 200 to 225...
Fetching jobs 225 to 250...
Fetching jobs 250 to 275...
Fetching jobs 275 to 300...
Fetching jobs 300 to 325...
Fetching jobs 325 to 350...
Fetching jobs 350 to 375...
Fetching jobs 375 to 400...
Fetching jobs 400 to 425...
Fetching jobs 425 to 450...
Fetching jobs 450 to 475...
Fetching jobs 475 to 500...
Fetching jobs 500 to 525...
Fetching jobs 525 to 550...
Fetching jobs 550 to 575...
Fetching jobs 575 to 600...
Fetching jobs 600 to 625...
Fetching jobs 625 to 650...
Fetching jobs 650 to 675...
Fetching jobs 675 to 700...
Fetching jobs 700 to 725...
Fetching jobs 725 to 750...
Fetching jobs 750 to 775...
Fetching jobs 775 to 800...
Fetching jobs 800 to 825...
Fetching jobs 825 to 850...
Fetching jobs 850 to 875...
Fetching jobs 875 to 900...


KeyboardInterrupt: 

In [ ]:
# HKEX

import requests
import time

def scrape_hkex_fixed_loop():
    session = requests.Session()
    base_url = "https://hkex.wd3.myworkdayjobs.com/HKEXCareerPage"
    api_url = "https://hkex.wd3.myworkdayjobs.com/wday/cxs/hkex/HKEXCareerPage/jobs"
    
    session.get(base_url)
    csrf_token = session.cookies.get("CALYPSO_CSRF_TOKEN")
    
    all_jobs = []
    limit = 20
    offset = 0
    total_available = None # Start as None to signal we haven't found it yet

    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "x-calypso-csrf-token": csrf_token
    }

    # We use a True loop and break manually based on our locked-in total
    while True:
        payload = {"appliedFacets": {}, "limit": limit, "offset": offset, "searchText": ""}
        
        try:
            response = session.post(api_url, json=payload, headers=headers)
            response.raise_for_status()
            data = response.json()

            # --- THE FIX: Only set total_available once ---
            if total_available is None:
                total_available = data.get("total", 0)
                print(f"🎯 Target Acquired: {total_available} jobs found.")

            batch = data.get("jobPostings", [])
            if not batch:
                print("📭 No more jobs returned in this batch.")
                break

            all_jobs.extend(batch)
            print(f"✅ Progress: {len(all_jobs)} / {total_available} (Offset {offset})")

            # --- BREAK CONDITION ---
            if len(all_jobs) >= total_available:
                break

            offset += limit
            time.sleep(1.5)

        except Exception as e:
            print(f"❌ Error: {e}")
            break

    print(f"\n✨ Final Count: {len(all_jobs)} jobs.")
    return all_jobs

# Run the full scrape
scrape_hkex_fixed_loop()

🎯 Target Acquired: 177 jobs found.
✅ Progress: 20 / 177 (Offset 0)
✅ Progress: 40 / 177 (Offset 20)
✅ Progress: 60 / 177 (Offset 40)
✅ Progress: 80 / 177 (Offset 60)
✅ Progress: 100 / 177 (Offset 80)
✅ Progress: 120 / 177 (Offset 100)
✅ Progress: 140 / 177 (Offset 120)
✅ Progress: 160 / 177 (Offset 140)
✅ Progress: 177 / 177 (Offset 160)

✨ Final Count: 177 jobs.


[{'title': 'Sr. Test Engineer - Associate - LMEC Testing - IT',
  'externalPath': '/job/CN-Shenzhen-HyQ/Sr-Test-Engineer---Associate---LMEC-Testing---IT_R003520',
  'locationsText': 'CN-Shenzhen-HyQ',
  'postedOn': 'Posted Today',
  'bulletFields': ['R003520']},
 {'title': 'AVP, Listing Enforcement',
  'externalPath': '/job/AVP--Listing-Enforcement_R002743',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R002743']},
 {'title': 'AVP, Listing Enforcement',
  'externalPath': '/job/HK-ONE-ES-43F/AVP--Listing-Enforcement_R003726',
  'locationsText': 'HK-ONE ES 43/F',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R003726']},
 {'title': 'Operation Officer (IPO Vetting) - 12 months contract',
  'externalPath': '/job/Operation-Officer--IPO-Vetting----12-months-contract_R003528',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R003528']},
 {'title': 'Modern Workspace EUC Technology Lead',
  'externalPath': '/job/UK-London/Modern-Workspace-EUC-Technology-Lead_R003781',
  'loc

In [56]:
import requests
from bs4 import BeautifulSoup

def scrape_bloomberg_avature():
    # The URL from your screenshot's link structure
    url = "https://bloomberg.avature.net/careers/SearchJobs"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9"
    }

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # --- TARGETING THE DOM FROM YOUR SCREENSHOT ---
    # We look for the <article> tags with the class 'article--result'
    job_cards = soup.select("article.article--result")
    
    print(f"🕵️ Avature Discovery: Found {len(job_cards)} job articles on this page.")

    for card in job_cards:
        # Navigate to the <h3> then the <a> tag
        link_tag = card.select_one("h3.article__header__text__title a.link")
        
        if link_tag:
            title = link_tag.get_text(strip=True)
            job_url = link_tag['href']
            
            # Subtitle usually contains location/department
            subtitle = card.select_one(".article__header__text__subtitle")
            location = subtitle.get_text(strip=True) if subtitle else "N/A"
            
            print(f"- {title} | {location}")
            print(f"  Link: {job_url}")
            print("-" * 30)

scrape_bloomberg_avature()

🕵️ Avature Discovery: Found 12 job articles on this page.
- Strategic Sourcing Manager (APAC), Singapore | Singapore, Singapore
  Link: https://bloomberg.avature.net/careers/JobDetail/Strategic-Sourcing-Manager-APAC-Singapore/18696
------------------------------
- East Asia Geoeconomics Analyst | Hong Kong, Hong Kong
  Link: https://bloomberg.avature.net/careers/JobDetail/East-Asia-Geoeconomics-Analyst/18697
------------------------------
- Regional Networks and Global Strategy - 6 Month Contract | London, United Kingdom
  Link: https://bloomberg.avature.net/careers/JobDetail/Regional-Networks-and-Global-Strategy-6-Month-Contract/18702
------------------------------
- Enterprise Services - Asset and Investment Manager (AIM) Enterprise Relationship Support | Tokyo, Japan
  Link: https://bloomberg.avature.net/careers/JobDetail/Enterprise-Services-Asset-and-Investment-Manager-AIM-Enterprise-Relationship-Support/18539
------------------------------
- Risk Manager - Engineering | London, Un

In [77]:
import requests
import time

def scrape_jpm_hong_kong_oracle_fixed():
    # The CORRECT endpoint for Oracle job lists
    api_url = "https://jpmc.fa.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
        "Ora-Api-Version": "1" # Mandatory for Oracle APIs
    }

    all_jobs = []
    offset = 0
    limit = 25

    print("🏦 Accessing JPM Oracle API with 'Finder' Logic...")

    while True:
        # THE FIX: Everything goes into this one specific string format
        finder_string = f"findReqs;siteNumber=CX_1001,limit={limit},locationId=300000000289330,offset={offset}"
        
        params = {
            "onlyData": "true",
            "expand": "requisitionList", # Tells Oracle to give us the actual jobs, not just links
            "finder": finder_string
        }

        try:
            print(f"📡 Fetching Offset {offset}...")
            response = requests.get(api_url, params=params, headers=headers)
            
            if response.status_code != 200:
                print(f"❌ Oracle rejected us with Status {response.status_code}: {response.text[:100]}")
                break

            data = response.json()
            
            # Oracle's JSON structure puts the results inside items[0]['requisitionList']
            items = data.get("items", [])
            if not items:
                print("🏁 No items returned from Oracle.")
                break
                
            job_list = items[0].get("requisitionList", [])
            
            if not job_list:
                print("🏁 No more jobs found in the requisition list. Scrape complete.")
                break

            for job in job_list:
                title = job.get("Title")
                job_id = job.get("Id")
                
                # Reconstruct the frontend URL for the master table
                url = f"https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/{job_id}"
                
                all_jobs.append({"title": title, "url": url})

            print(f"✅ Offset {offset}: Added {len(job_list)} jobs.")

            # Check if we've hit the end of the line
            if len(job_list) < limit:
                break
                
            offset += limit
            time.sleep(1) 

        except Exception as e:
            print(f"❌ Scrape Failed: {e}")
            break

    print(f"\n✨ Done! Extracted {len(all_jobs)} Hong Kong jobs from JPM.")
    return all_jobs

scrape_jpm_hong_kong_oracle_fixed()

🏦 Accessing JPM Oracle API with 'Finder' Logic...
📡 Fetching Offset 0...
✅ Offset 0: Added 25 jobs.
📡 Fetching Offset 25...
✅ Offset 25: Added 25 jobs.
📡 Fetching Offset 50...
✅ Offset 50: Added 25 jobs.
📡 Fetching Offset 75...
✅ Offset 75: Added 25 jobs.
📡 Fetching Offset 100...
✅ Offset 100: Added 25 jobs.
📡 Fetching Offset 125...
✅ Offset 125: Added 9 jobs.

✨ Done! Extracted 134 Hong Kong jobs from JPM.


[{'title': 'Market Asset Servicing - Risk Analyst - Vice President',
  'url': 'https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/210694184'},
 {'title': 'Securities Services - Asia Pacific Pitchbooks Writer – Proposals & Presentations Officer, Associate',
  'url': 'https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/210705362'},
 {'title': 'Global Investment Banking - TMT- Analyst/Associate (Korean - Speaking)',
  'url': 'https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/210716079'},
 {'title': 'Equities Settlement - Analyst/Associate',
  'url': 'https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/210719940'},
 {'title': 'APAC Markets Sales Business Control Manager - Executive Director',
  'url': 'https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/210732011'},
 {'title': 'Hong Kong Location Control Management - Executive Director',
  'url': 'https:

In [78]:
import pandas as pd
import requests
import time

def fetch_greenhouse_jobs(company_name, api_url):
    """Parses Greenhouse API endpoints."""
    jobs_list = []
    response = requests.get(api_url)
    
    if response.status_code == 200:
        data = response.json()
        jobs = data.get('jobs', [])
        for job in jobs:
            jobs_list.append({
                'Company': company_name,
                'Job Title': job.get('title'),
                'Location': job.get('location', {}).get('name', 'Unknown'),
                'Job URL': job.get('absolute_url'),
                'Department': job.get('departments', [{'name': ''}])[0].get('name', ''),
                'Updated At': job.get('updated_at')
            })
    else:
        print(f"Failed to fetch {company_name}. Status Code: {response.status_code}")
        
    return jobs_list

def fetch_generic_jobs(company_name, api_url):
    """Fallback parser for unknown API structures."""
    jobs_list = []
    response = requests.get(api_url)
    
    if response.status_code == 200:
        try:
            data = response.json()
            # If it's a direct list of jobs (e.g., Lever often returns a list directly)
            if isinstance(data, list):
                jobs = data
            elif isinstance(data, dict):
                # Guess where the jobs are stored
                if 'jobs' in data: jobs = data['jobs']
                elif 'data' in data: jobs = data['data']
                else: jobs = []
            
            for job in jobs:
                jobs_list.append({
                    'Company': company_name,
                    'Job Title': job.get('text') or job.get('title') or job.get('name', 'Unknown'),
                    'Location': str(job.get('location') or job.get('categories', {}).get('location', 'Unknown')),
                    'Job URL': job.get('hostedUrl') or job.get('applyUrl') or job.get('absolute_url', 'Unknown'),
                    'Department': str(job.get('categories', {}).get('team', '')),
                    'Updated At': job.get('createdAt') or job.get('updated_at', '')
                })
        except Exception as e:
            print(f"Error parsing generic JSON for {company_name}: {e}")
    else:
        print(f"Failed to fetch {company_name}. Status Code: {response.status_code}")
        
    return jobs_list

def main():
    # 1. Load the target companies CSV
    input_csv = "target_companies_api.csv"
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: Could not find {input_csv}.")
        return

    # 2. Filter for rows that actually have a valid API Endpoint
    valid_apis = df[df['API Endpoint'].notna() & (df['API Endpoint'] != 'N/A')]
    
    print(f"Found {len(valid_apis)} companies with valid API endpoints.")
    
    all_jobs = []

    # 3. Iterate through each valid company and scrape
    for index, row in valid_apis.iterrows():
        company = row['Company']
        platform = str(row['Platform']).lower()
        api_url = row['API Endpoint']
        
        print(f"Scraping jobs for {company} via {platform.title()} API...")
        
        # Route to the correct parsing logic based on the platform
        if 'greenhouse' in platform:
            jobs = fetch_greenhouse_jobs(company, api_url)
        else:
            # Fallback for Lever, Workday JSONs, or custom APIs
            jobs = fetch_generic_jobs(company, api_url)
            
        print(f"-> Extracted {len(jobs)} jobs for {company}.")
        all_jobs.extend(jobs)
        
        # Be polite to the APIs, add a small delay
        time.sleep(1)

    # 4. Consolidate and export the data
    if all_jobs:
        output_df = pd.DataFrame(all_jobs)
        output_file = "master_jobs_list.csv"
        output_df.to_csv(output_file, index=False)
        print(f"\n✅ Success! Scraped {len(all_jobs)} total jobs.")
        print(f"Data successfully saved to '{output_file}'.")
    else:
        print("\nNo jobs could be extracted.")

if __name__ == "__main__":
    main()

Found 13 companies with valid API endpoints.
Scraping jobs for Crypto.com via Lever API...
-> Extracted 43 jobs for Crypto.com.
Scraping jobs for OKX via Greenhouse API...
-> Extracted 305 jobs for OKX.
Scraping jobs for Airwallex via Ashby API...
-> Extracted 622 jobs for Airwallex.
Scraping jobs for Reap via Teamtailor API...


KeyboardInterrupt: 